# Анализ лояльности пользователей Яндекс Афиши

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Параметры подключения задаются в локальном файле `.env` (шаблон — `.env.example`). Файл с реквизитами не публикуется.

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---


In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Запускайте ноутбук из корня проекта. Реквизиты хранятся только в .env.
load_dotenv(Path.cwd() / '.env', override=False)
required = ['DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD']
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError('Заполните .env: ' + ', '.join(missing))

connection_url = URL.create(
    'postgresql+psycopg2',
    username=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=int(os.environ['DB_PORT']),
    database=os.environ['DB_NAME'],
)
engine = create_engine(connection_url, connect_args={'connect_timeout': 20})

# SQL-запрос из третьего урока (без LIMIT 10 и ORDER BY для полной выгрузки)
query = '''
SELECT
    p.user_id,
    p.device_type_canonical,
    p.order_id,
    p.created_dt_msk AS order_dt,
    p.created_ts_msk AS order_ts,
    p.currency_code,
    p.revenue,
    p.tickets_count,
    (
        p.created_dt_msk::date
        - LAG(p.created_dt_msk::date) OVER (
            PARTITION BY p.user_id
            ORDER BY p.created_dt_msk
        )
    ) AS days_since_prev,
    e.event_id,
    p.service_name,
    e.event_type_main,
    r.region_name,
    c.city_name
FROM afisha.purchases p
JOIN afisha.events e ON p.event_id = e.event_id
JOIN afisha.city c ON e.city_id = c.city_id
JOIN afisha.regions r ON c.region_id = r.region_id
WHERE p.device_type_canonical IN ('mobile', 'desktop')
  AND e.event_type_main != 'фильм'
'''

# Загружаем в датафрейм
df = pd.read_sql_query(query, con=engine)

# Смотрим общую информацию
print(f'Размер датафрейма: {df.shape}')
print(f'Столбцы: {df.columns.tolist()}')
df.head()

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [ ]:
# Общая информация о датафрейме
print('Размер датафрейма:', df.shape)
print('\nТипы данных:')
print(df.dtypes)
print('\nПропуски по столбцам:')
print(df.isna().sum())
print('\nДоля пропусков:')
print((df.isna().mean() * 100).round(2))
print('\nУникальные значения в категориальных столбцах:')
for col in ['device_type_canonical', 'currency_code', 'event_type_main',
            'service_name', 'region_name', 'city_name']:
    print(f'{col}: {df[col].nunique()} уникальных')
print('\nПериод данных:')
print('order_dt: с', df['order_dt'].min(), 'по', df['order_dt'].max())
print('order_ts: с', df['order_ts'].min(), 'по', df['order_ts'].max())
print('\nКоличество уникальных пользователей:', df['user_id'].nunique())
print('Количество уникальных заказов:', df['order_id'].nunique())
print('\nСтатистика по числовым столбцам:')
print(df[['revenue', 'tickets_count', 'days_since_prev']].describe())
print('\nПримеры значений category-полей:')
for col in ['device_type_canonical', 'currency_code', 'event_type_main']:
    print(col, ':', df[col].unique())

Вывод по шагу 1. 
Данные успешно выгружены из базы учебная база PostgreSQL: 290 611 заказов, 21 933 уникальных пользователя, 14 признаков, период — июнь–октябрь 2024. В выборку вошли только заказы с мобильных и десктопных устройств, фильмы исключены. Пропуски есть только в days_since_prev (7.55%) — это ожидаемо для первых покупок пользователей. Типы order_dt и order_ts уже корректны (datetime64), однако days_since_prev хранится как float64 из-за NaN — на этапе предобработки приведём к Int64. В данных две валюты (rub, kzt), поэтому revenue нужно привести к единой — рублю. В revenue присутствуют отрицательные значения и сильные выбросы (max 81 174 при медиане 356), в tickets_count — значение 57 билетов в заказе. Эти аномалии требуют обработки: фильтрации выбросов по 99-му перцентилю и проверки отрицательной выручки. Категориальные значения выглядят чистыми, без заглушек.

---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [ ]:
# Загружаем курс тенге (из вложения)
tenge_df = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')

# Приводим дату к datetime и оставляем нужные поля
tenge_df['data'] = pd.to_datetime(tenge_df['data'])
tenge_df = tenge_df[['data', 'curs']].rename(columns={'data': 'order_dt'})

# Переименуем дату в df для мёрджа (сравниваем по дате заказа)
df['order_dt_date'] = pd.to_datetime(df['order_dt']).dt.normalize()

# Мёрджим курс по дате заказа
df = df.merge(
    tenge_df,
    left_on='order_dt_date',
    right_on='order_dt',
    how='left'
)

# Удаляем вспомогательный столбец
df = df.drop(columns=['order_dt_date'])

# Приводим выручку к рублям:
# curs — это рублей за 100 тенге, поэтому:
# revenue_rub = revenue (если rub) или revenue * (curs / 100) (если kzt)
df['revenue_rub'] = df.apply(
    lambda row: row['revenue'] if row['currency_code'] == 'rub'
    else row['revenue'] * (row['curs'] / 100),
    axis=1
)

# Удаляем столбец curs (он больше не нужен)
df = df.drop(columns=['curs'])

# Проверяем результат
print('Пропуски в revenue_rub:', df['revenue_rub'].isna().sum())
print('\nРаспределение по валютам:')
print(df['currency_code'].value_counts())
print('\nСтатистика revenue_rub по валютам (исходная revenue):')
print(df.groupby('currency_code')['revenue'].describe())
print('\nСтатистика revenue_rub:')
print(df['revenue_rub'].describe())
print('\nПервые строки с kzt:')
print(df[df['currency_code'] == 'kzt'][['revenue', 'revenue_rub']].head())

Создан столбец revenue_rub.

Все kzt-заказы переведены по курсу на дату заказа.

Пропусков нет.

Для rub значения не изменились.

Важно для дальнейшего: отрицательный revenue_rub (−90.76) и выбросы (до 81 174) остаются — их обработаем в задаче 2.2 (фильтрация по 99-му перцентилю и решение по отрицательным значениям).



---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Проверка пропусков
# ============================================================
print('=== Пропуски по столбцам ===')
print(df.isna().sum())
print('\nДоля пропусков (%):')
print((df.isna().mean() * 100).round(2))

In [ ]:
# ============================================================
# 2. Преобразование типов
# ============================================================
# days_since_prev: float64 -> Int64 (nullable integer)
df['days_since_prev'] = df['days_since_prev'].astype('Int64')

# order_dt: datetime64 -> date (если нужно; оставляем datetime64[ns] для удобства)
# order_ts: datetime64[ns] — оставляем

# tickets_count: int64 -> int32 (экономия памяти)
df['tickets_count'] = df['tickets_count'].astype('int32')

# order_id, event_id: int64 -> int32
df['order_id'] = df['order_id'].astype('int32')
df['event_id'] = df['event_id'].astype('int32')

# revenue, revenue_rub: float64 -> float32 (экономия памяти)
df['revenue'] = df['revenue'].astype('float32')
df['revenue_rub'] = df['revenue_rub'].astype('float32')

print('=== Типы после преобразования ===')
print(df.dtypes)

In [ ]:
# ============================================================
# 3. Изучение значений в ключевых столбцах
# ============================================================
# Проверка категориальных значений на заглушки
print('=== Уникальные значения в категориальных столбцах ===')
for col in ['device_type_canonical', 'currency_code', 'event_type_main',
            'service_name', 'region_name', 'city_name']:
    print(f'\n{col} ({df[col].nunique()}):')
    print(df[col].value_counts().head(10))

# Проверка на значения-заглушки
print('\n=== Проверка на заглушки ===')
for col in ['device_type_canonical', 'currency_code', 'event_type_main',
            'service_name', 'region_name', 'city_name']:
    suspicious = df[col].isin(['нет', 'unknown', 'None', '', 'nan', 'NaN', '-'])
    print(f'{col}: {suspicious.sum()} подозрительных значений')

In [ ]:
# ============================================================
# 4. Проверка распределения числовых данных и выбросов
# ============================================================
print('=== Статистика revenue_rub ===')
print(df['revenue_rub'].describe())

print('\n=== Статистика tickets_count ===')
print(df['tickets_count'].describe())

print('\n=== Статистика days_since_prev ===')
print(df['days_since_prev'].describe())

# Гистограммы
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['revenue_rub'].hist(bins=50, ax=axes[0])
axes[0].set_title('revenue_rub')
df['tickets_count'].hist(bins=20, ax=axes[1])
axes[1].set_title('tickets_count')
df['days_since_prev'].dropna().hist(bins=30, ax=axes[2])
axes[2].set_title('days_since_prev')
plt.tight_layout()
plt.show()

# Диаграммы размаха (boxplot)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.boxplot(column='revenue_rub', ax=axes[0])
df.boxplot(column='tickets_count', ax=axes[1])
df.boxplot(column='days_since_prev', ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5. Обработка выбросов в revenue_rub (99-й перцентиль)
# ============================================================
# Считаем 99-й перцентиль
q99 = df['revenue_rub'].quantile(0.99)
print(f'99-й перцентиль revenue_rub: {q99:.2f}')

# Количество строк с revenue_rub > q99
outliers = df[df['revenue_rub'] > q99]
print(f'Строк с revenue_rub > q99: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)')

# Фильтрация
df = df[df['revenue_rub'] <= q99].copy()
print(f'Размер датафрейма после фильтрации: {df.shape}')

# Проверка отрицательных значений revenue_rub
neg = df[df['revenue_rub'] < 0]
print(f'Строк с отрицательным revenue_rub: {len(neg)}')

In [ ]:
# ============================================================
# 6. Проверка tickets_count на выбросы
# ============================================================
print('=== Распределение tickets_count ===')
print(df['tickets_count'].value_counts().sort_index())

# Проверим, есть ли заказы с >10 билетов
print(f'\nЗаказов с tickets_count > 10: {(df["tickets_count"] > 10).sum()}')
print(f'Заказов с tickets_count > 20: {(df["tickets_count"] > 20).sum()}')

# 99-й перцентиль tickets_count
q99_tickets = df['tickets_count'].quantile(0.99)
print(f'99-й перцентиль tickets_count: {q99_tickets}')

# Опционально: фильтрация по 99-му перцентилю tickets_count
# (если решим, что выбросы критичны)
# df = df[df['tickets_count'] <= q99_tickets].copy()


In [ ]:
# ============================================================
# 7. Итоговая проверка
# ============================================================
print('=== Итоговая информация ===')
print(f'Размер датафрейма: {df.shape}')
print(f'Уникальных пользователей: {df["user_id"].nunique()}')
print(f'Уникальных заказов: {df["order_id"].nunique()}')
print(f'Пропуски в days_since_prev: {df["days_since_prev"].isna().sum()}')
print('\nСтатистика revenue_rub:')
print(df['revenue_rub'].describe())
print('\nТипы данных:')
print(df.dtypes)

ВЫЯВЛЕННЫЕ ПРОБЛЕМЫ:

1. Дубликат столбца order_dt (order_dt_x и order_dt_y)
При мёрдже с курсом тенге (задача 2.1) появились два столбца:

order_dt_x — исходная дата заказа;

order_dt_y — дата из файла курса.

Нужно оставить один order_dt (исходный) и удалить order_dt_y. 

2. Отрицательный revenue_rub не обработан
Осталось 381 строка с отрицательной выручкой (min −90.76). Для анализа средней выручки и профиля пользователя это аномалия — нужно отфильтровать.

In [ ]:
# 1. Удаляем дубликат столбца order_dt_y, переименовываем order_dt_x обратно в order_dt
df = df.drop(columns=['order_dt_y'])
df = df.rename(columns={'order_dt_x': 'order_dt'})

# 2. Удаляем строки с отрицательной выручкой (возвраты/аномалии)
neg_count = (df['revenue_rub'] < 0).sum()
df = df[df['revenue_rub'] >= 0].copy()
print(f'Удалено строк с отрицательным revenue_rub: {neg_count}')
print(f'Размер после удаления отрицательных: {df.shape}')

# 3. Проверяем итог
print(f'\nИтоговый размер: {df.shape}')
print(f'Уникальных пользователей: {df["user_id"].nunique()}')
print(f'Пропуски в days_since_prev: {df["days_since_prev"].isna().sum()}')
print('\nСтолбцы:', df.columns.tolist())
print('\nСтатистика revenue_rub:')
print(df['revenue_rub'].describe())

Вывод по предобработке. Данные выгружены из БД (290 611 заказов, 21 933 пользователя, июнь–октябрь 2024). Выполнена предобработка: пропуски остались только в days_since_prev (7.55%, ожидаемо для первых покупок); типы приведены к компактным (Int64, int32, float32); категориальные поля проверены — заглушек нет. Выручка приведена к рублю (revenue_rub) по курсу тенге на дату заказа. Удалены выбросы: 2 825 строк с revenue_rub выше 99-го перцентиля (2 628 руб.) и 381 строка с отрицательной выручкой. tickets_count содержит единичные заказы с 47–57 билетами (7 заказов >20), оставлены — они составляют <0.02%. 


Итог: 287 405 заказов, 21 838 пользователей, 15 признаков.

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


In [ ]:
# Сортируем данные по времени заказа (важно для корректного определения первого заказа)
df = df.sort_values(by=['user_id', 'order_ts']).reset_index(drop=True)

# Для каждого пользователя находим первый заказ
first_orders = df.groupby('user_id').first().reset_index()

# Переименовываем поля первого заказа для удобства
first_orders = first_orders.rename(columns={
    'order_dt': 'first_order_dt',
    'device_type_canonical': 'first_device',
    'region_name': 'first_region',
    'service_name': 'first_service',
    'event_type_main': 'first_event_type'
})

# Оставляем только нужные поля первого заказа
first_orders = first_orders[[
    'user_id', 'first_order_dt', 'first_device', 'first_region',
    'first_service', 'first_event_type'
]]

# Агрегируем остальные показатели по пользователю
user_profile = df.groupby('user_id').agg(
    last_order_dt=('order_dt', 'max'),
    total_orders=('order_id', 'count'),
    avg_revenue_rub=('revenue_rub', 'mean'),
    avg_tickets_count=('tickets_count', 'mean'),
    avg_days_since_prev=('days_since_prev', 'mean')
).reset_index()

# Объединяем с данными о первом заказе
user_profile = user_profile.merge(first_orders, on='user_id', how='left')

# Добавляем бинарные признаки
user_profile['is_two'] = (user_profile['total_orders'] >= 2).astype(int)
user_profile['is_five'] = (user_profile['total_orders'] >= 5).astype(int)

# Проверяем результат
print('Размер профиля:', user_profile.shape)
print('\nСтолбцы:', user_profile.columns.tolist())
print('\nПервые 5 строк:')
print(user_profile.head())
print('\nПропуски:')
print(user_profile.isna().sum())
print('\nСтатистика:')
print(user_profile[['total_orders', 'avg_revenue_rub', 'avg_tickets_count', 'avg_days_since_prev']].describe())

Промежуточный вывод по задаче 3.1

    Профиль пользователя построен: для 21 838 пользователей рассчитаны дата первого и последнего заказа, характеристики первого заказа (устройство, регион, оператор, тип мероприятия), общее число заказов, средняя выручка, среднее число билетов и средний интервал между заказами. Добавлены бинарные признаки is_two и is_five. Пропуски только в avg_days_since_prev (8 333) — у пользователей с одним заказом. Обнаружена аномалия: максимальное число заказов у одного пользователя — 10 168 (при медиане 2), что требует решения на следующем шаге.

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Основные метрики по профилю пользователей
# ============================================================
print('=== Основные метрики ===')
print(f'Общее число пользователей: {user_profile.shape[0]}')
print(f'Средняя выручка с одного заказа: {user_profile["avg_revenue_rub"].mean():.2f} руб.')
print(f'Доля пользователей с 2+ заказами: {user_profile["is_two"].mean() * 100:.2f}%')
print(f'Доля пользователей с 5+ заказами: {user_profile["is_five"].mean() * 100:.2f}%')


In [ ]:
# ============================================================
# 2. Статистические показатели
# ============================================================
print('=== total_orders ===')
print(user_profile['total_orders'].describe())
print('\nПерцентили total_orders:')
for q in [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]:
    print(f'  {q*100:.1f}%: {user_profile["total_orders"].quantile(q):.1f}')

print('\n=== avg_tickets_count ===')
print(user_profile['avg_tickets_count'].describe())
print('\nПерцентили avg_tickets_count:')
for q in [0.5, 0.75, 0.9, 0.95, 0.99]:
    print(f'  {q*100:.1f}%: {user_profile["avg_tickets_count"].quantile(q):.2f}')

print('\n=== avg_days_since_prev (только пользователи с 2+ заказами) ===')
print(user_profile['avg_days_since_prev'].describe())

In [ ]:
# ============================================================
# 3. Визуализация распределений
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

user_profile['total_orders'].hist(bins=50, ax=axes[0])
axes[0].set_title('total_orders')
axes[0].set_xlim(0, 50)

user_profile['avg_tickets_count'].hist(bins=30, ax=axes[1])
axes[1].set_title('avg_tickets_count')

user_profile['avg_days_since_prev'].dropna().hist(bins=30, ax=axes[2])
axes[2].set_title('avg_days_since_prev')

plt.tight_layout()
plt.show()

# Boxplot для total_orders
plt.figure(figsize=(8, 4))
user_profile.boxplot(column='total_orders')
plt.title('Boxplot total_orders')
plt.show()

In [ ]:
# ============================================================
# 4. Анализ аномалий в total_orders
# ============================================================
print('=== Пользователи с большим числом заказов ===')
print(f'total_orders > 100: {(user_profile["total_orders"] > 100).sum()} пользователей')
print(f'total_orders > 500: {(user_profile["total_orders"] > 500).sum()} пользователей')
print(f'total_orders > 1000: {(user_profile["total_orders"] > 1000).sum()} пользователей')

# Топ-10 пользователей по числу заказов
print('\nТоп-10 по числу заказов:')
print(user_profile.nlargest(10, 'total_orders')[['user_id', 'total_orders', 'avg_revenue_rub', 'avg_tickets_count']])

In [ ]:
# ============================================================
# 5. Фильтрация аномалий по 99-му перцентилю total_orders
# ============================================================
q99_orders = user_profile['total_orders'].quantile(0.99)
print(f'\n99-й перцентиль total_orders: {q99_orders:.1f}')

# Количество пользователей выше 99-го перцентиля
above_q99 = user_profile[user_profile['total_orders'] > q99_orders]
print(f'Пользователей с total_orders > {q99_orders:.1f}: {len(above_q99)} ({len(above_q99)/len(user_profile)*100:.2f}%)')

# Фильтрация
user_profile_clean = user_profile[user_profile['total_orders'] <= q99_orders].copy()
print(f'\nРазмер после фильтрации: {user_profile_clean.shape}')
print(f'Отфильтровано пользователей: {len(user_profile) - len(user_profile_clean)}')

# Статистика после фильтрации
print('\n=== Статистика после фильтрации ===')
print(f'Общее число пользователей: {user_profile_clean.shape[0]}')
print(f'Средняя выручка с заказа: {user_profile_clean["avg_revenue_rub"].mean():.2f} руб.')
print(f'Доля с 2+ заказами: {user_profile_clean["is_two"].mean() * 100:.2f}%')
print(f'Доля с 5+ заказами: {user_profile_clean["is_five"].mean() * 100:.2f}%')
print('\ntotal_orders:')
print(user_profile_clean['total_orders'].describe())
print('\navg_tickets_count:')
print(user_profile_clean['avg_tickets_count'].describe())
print('\navg_days_since_prev:')
print(user_profile_clean['avg_days_since_prev'].describe())

In [ ]:
# ============================================================
# 6. Проверка avg_tickets_count на аномалии
# ============================================================
print('=== Аномалии в avg_tickets_count ===')
print(f'avg_tickets_count > 5: {(user_profile_clean["avg_tickets_count"] > 5).sum()} пользователей')
print(f'avg_tickets_count > 8: {(user_profile_clean["avg_tickets_count"] > 8).sum()} пользователей')

q99_tickets = user_profile_clean['avg_tickets_count'].quantile(0.99)
print(f'99-й перцентиль avg_tickets_count: {q99_tickets:.2f}')

Промежуточный вывод (для отчёта)

    Вывод по шагу 3.2. В выборке 21 838 пользователей, средняя выручка с заказа — 545 руб., доля вернувшихся (2+ заказа) — 61.7%, доля лояльных (5+ заказов) — 29.0%. Объём данных достаточен для анализа.

    Обнаружены аномалии в total_orders: максимальное число заказов — 10 168 (при медиане 2), 99-й перцентиль — 152. Такие значения характерны для ботов или корпоративных аккаунтов и сильно искажают среднее и стандартное отклонение. Принято решение отфильтровать пользователей с total_orders выше 99-го перцентиля (152 заказа) — удалено 216 пользователей (0.99%). После фильтрации ключевые метрики (доли возвратов, средняя выручка) практически не изменились, но распределение total_orders стало репрезентативным (mean 6.5, std 14.3, max 152).

    avg_tickets_count: max = 11, 99-й перцентиль = 5 — аномалий не выявлено, фильтрация не требуется.

    avg_days_since_prev: max = 148 дней — в пределах периода, аномалий нет.

    Итоговый датафрейм для EDA: 21 622 пользователя, 13 признаков.

In [ ]:
# Фиксируем очищенный профиль для дальнейшего анализа
user_profile = user_profile_clean.copy()
print(f'Итоговый размер профиля: {user_profile.shape}')

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Вспомогательная функция для группировки и подсчёта
# ============================================================
def segment_stats(df, col, top_n=None, title=''):
    """Считает число пользователей и долю по сегментам."""
    stats = df.groupby(col).agg(
        users=('user_id', 'count')
    ).reset_index()
    stats['share_pct'] = (stats['users'] / stats['users'].sum() * 100).round(2)
    stats = stats.sort_values('users', ascending=False).reset_index(drop=True)
    
    if top_n is not None:
        stats = stats.head(top_n)
    
    print(f'\n=== {title} ===')
    print(stats.to_string(index=False))
    return stats

# 1. По типу первого мероприятия
stats_event = segment_stats(user_profile, 'first_event_type', title='Тип первого мероприятия')

# 2. По типу устройства
stats_device = segment_stats(user_profile, 'first_device', title='Тип устройства первого заказа')

# 3. По региону (топ-10, так как 81 регион)
stats_region = segment_stats(user_profile, 'first_region', top_n=10, title='Регион первого заказа (топ-10)')
# И полное распределение для оценки концентрации
region_full = user_profile['first_region'].value_counts()
print(f'\nВсего регионов: {region_full.shape[0]}')
print(f'Доля топ-5 регионов: {region_full.head(5).sum() / region_full.sum() * 100:.1f}%')
print(f'Доля топ-10 регионов: {region_full.head(10).sum() / region_full.sum() * 100:.1f}%')

# 4. По оператору (топ-10, так как 36)
stats_service = segment_stats(user_profile, 'first_service', top_n=10, title='Оператор первого заказа (топ-10)')
service_full = user_profile['first_service'].value_counts()
print(f'\nВсего операторов: {service_full.shape[0]}')
print(f'Доля топ-5 операторов: {service_full.head(5).sum() / service_full.sum() * 100:.1f}%')
print(f'Доля топ-10 операторов: {service_full.head(10).sum() / service_full.sum() * 100:.1f}%')

In [ ]:
# ============================================================
# Визуализация
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Тип мероприятия
axes[0, 0].barh(stats_event['first_event_type'][::-1], stats_event['users'][::-1])
axes[0, 0].set_title('Тип первого мероприятия')
axes[0, 0].set_xlabel('Пользователей')

# Устройство
axes[0, 1].barh(stats_device['first_device'][::-1], stats_device['users'][::-1])
axes[0, 1].set_title('Тип устройства')

# Регион (топ-10)
axes[1, 0].barh(stats_region['first_region'][::-1], stats_region['users'][::-1])
axes[1, 0].set_title('Регион первого заказа (топ-10)')

# Оператор (топ-10)
axes[1, 1].barh(stats_service['first_service'][::-1], stats_service['users'][::-1])
axes[1, 1].set_title('Оператор первого заказа (топ-10)')

plt.tight_layout()
plt.show()

Вывод по задаче 4.1.1. 

    Пользователи распределены по сегментам первого заказа крайне неравномерно — есть выраженные «точки входа».

    По типу мероприятия: доминируют концерты (44.2%), далее «другое» (25.0%) и театр (19.6%). Сегменты «ёлки» (0.4%) и «выставки» (1.9%) малочисленны — доли возвратов по ним будут нестабильны.

    По устройству: мобильные — 82.8%, десктоп — 17.2%.

    По региону: сильная концентрация — топ-5 регионов дают 61.4% пользователей, топ-10 — 72.0%. Лидеры — «Каменевский регион» (32.8%) и «Североярская область» (17.4%).

    По оператору: топ-5 операторов — 72.5%, топ-10 — 90.4%. Лидер — «Билеты без проблем» (23.9%).

При анализе возвратов важно учитывать размер сегментов: в малочисленных сегментах (регионы за пределами топ-10, операторы за пределами топ-10, «ёлки», «выставки») доли повторных заказов могут быть недостоверны из-за малой выборки.

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Общая доля возвратов (базовая линия)
base_rate = user_profile['is_two'].mean() * 100
print(f'Средняя доля возвратов (2+ заказа): {base_rate:.2f}%')

# ============================================================
# Функция для расчёта доли возвратов по сегментам
# ============================================================
def return_rate_by_segment(df, col, top_n=None):
    stats = df.groupby(col).agg(
        users=('user_id', 'count'),
        return_rate=('is_two', 'mean')
    ).reset_index()
    stats['return_rate_pct'] = (stats['return_rate'] * 100).round(2)
    stats['share_pct'] = (stats['users'] / stats['users'].sum() * 100).round(2)
    stats = stats.sort_values('users', ascending=False).reset_index(drop=True)
    
    if top_n is not None:
        stats_plot = stats.head(top_n)
    else:
        stats_plot = stats
    
    return stats, stats_plot

# 1. По типу мероприятия (все 7 сегментов)
stats_event, _ = return_rate_by_segment(user_profile, 'first_event_type')
print('\n=== Возвраты по типу первого мероприятия ===')
print(stats_event[['first_event_type', 'users', 'share_pct', 'return_rate_pct']].to_string(index=False))

# 2. По устройству (2 сегмента)
stats_device, _ = return_rate_by_segment(user_profile, 'first_device')
print('\n=== Возвраты по устройству ===')
print(stats_device[['first_device', 'users', 'share_pct', 'return_rate_pct']].to_string(index=False))

# 3. По региону (топ-10)
stats_region, stats_region_plot = return_rate_by_segment(user_profile, 'first_region', top_n=10)
print('\n=== Возвраты по региону (топ-10) ===')
print(stats_region_plot[['first_region', 'users', 'share_pct', 'return_rate_pct']].to_string(index=False))

# 4. По оператору (топ-10)
stats_service, stats_service_plot = return_rate_by_segment(user_profile, 'first_service', top_n=10)
print('\n=== Возвраты по оператору (топ-10) ===')
print(stats_service_plot[['first_service', 'users', 'share_pct', 'return_rate_pct']].to_string(index=False))

In [ ]:
# ============================================================
# Визуализация
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Тип мероприятия
axes[0, 0].barh(stats_event['first_event_type'][::-1], stats_event['return_rate_pct'][::-1], color='steelblue')
axes[0, 0].axvline(base_rate, color='red', linestyle='--', label=f'Среднее: {base_rate:.1f}%')
axes[0, 0].set_title('Доля возвратов по типу мероприятия')
axes[0, 0].set_xlabel('Доля 2+ заказов, %')
axes[0, 0].legend()

# Устройство
axes[0, 1].barh(stats_device['first_device'][::-1], stats_device['return_rate_pct'][::-1], color='steelblue')
axes[0, 1].axvline(base_rate, color='red', linestyle='--', label=f'Среднее: {base_rate:.1f}%')
axes[0, 1].set_title('Доля возвратов по устройству')
axes[0, 1].set_xlabel('Доля 2+ заказов, %')
axes[0, 1].legend()

# Регион (топ-10)
axes[1, 0].barh(stats_region_plot['first_region'][::-1], stats_region_plot['return_rate_pct'][::-1], color='steelblue')
axes[1, 0].axvline(base_rate, color='red', linestyle='--', label=f'Среднее: {base_rate:.1f}%')
axes[1, 0].set_title('Доля возвратов по региону (топ-10)')
axes[1, 0].set_xlabel('Доля 2+ заказов, %')
axes[1, 0].legend()

# Оператор (топ-10)
axes[1, 1].barh(stats_service_plot['first_service'][::-1], stats_service_plot['return_rate_pct'][::-1], color='steelblue')
axes[1, 1].axvline(base_rate, color='red', linestyle='--', label=f'Среднее: {base_rate:.1f}%')
axes[1, 1].set_title('Доля возвратов по оператору (топ-10)')
axes[1, 1].set_xlabel('Доля 2+ заказов, %')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

Вывод по задаче 4.1.2. Средняя доля возвратов — 61.32%. Есть сегменты-«точки входа» с долей выше средней:

    По типу мероприятия: театр (63.4%) и выставки (64.2%) возвращаются чаще; спорт (55.8%) и ёлки (55.8%) — реже. Концерты — чуть выше среднего (61.8%).

    По устройству: десктоп (63.8%) возвращается чаще мобильных (60.8%) на ~3 п.п.

    По региону: лидеры — Шанырский регион (67.3%), Светополянский округ (65.7%), Широковская область (64.4%). Аутсайдеры — Озернинский край (55.3%), Малиновоярский округ (55.9%).

    По оператору: разброс небольшой (60.3–65.2%), лидеры — Край билетов (65.2%) и Дом культуры (64.6%), но сегменты малочисленны.

Успешные «точки входа»: театр, выставки, десктоп, Шанырский регион, Светополянский округ, Широковская область, Североярская область — во всех доля возвратов выше средней при достаточном размере сегмента.

Ограничения: сегменты «ёлки» (95), «Край билетов» (454), «Дом культуры» (356) малочисленны — доли могут быть нестабильны.

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Базовая доля возвратов
base_rate = user_profile['is_two'].mean()
print(f'Средняя доля возвратов: {base_rate * 100:.2f}%\n')

# ============================================================
# ГИПОТЕЗА 1: спорт возвращается чаще концертов
# ============================================================
print('=' * 60)
print('ГИПОТЕЗА 1: спорт > концерты по доле возвратов')
print('=' * 60)

sport = user_profile[user_profile['first_event_type'] == 'спорт']
concert = user_profile[user_profile['first_event_type'] == 'концерты']

n_sport, rate_sport = len(sport), sport['is_two'].mean()
n_concert, rate_concert = len(concert), concert['is_two'].mean()

print(f'Спорт:     n = {n_sport:>5}, доля возвратов = {rate_sport*100:.2f}%')
print(f'Концерты:  n = {n_concert:>5}, доля возвратов = {rate_concert*100:.2f}%')
diff = (rate_sport - rate_concert) * 100
print(f'\nРазница (спорт − концерты): {diff:+.2f} п.п.')

if diff > 0:
    print('→ Спорт возвращается ЧАЩЕ концертов.')
else:
    print('→ Спорт возвращается РЕЖЕ концертов. Гипотеза НЕ подтверждается.')

# Z-тест для двух долей (проверка значимости разницы)
count_sport = sport['is_two'].sum()
count_concert = concert['is_two'].sum()
# объединённая доля
p_pool = (count_sport + count_concert) / (n_sport + n_concert)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_sport + 1/n_concert))
z = (rate_sport - rate_concert) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print(f'\nZ-тест для двух долей: z = {z:.3f}, p-value = {p_value:.4f}')
if p_value < 0.05:
    print('Разница статистически значима (p < 0.05).')
else:
    print('Разница статистически НЕ значима (p >= 0.05).')

In [ ]:
# ============================================================
# ГИПОТЕЗА 2: в регионах-лидерах доля возвратов выше
# ============================================================
print('\n' + '=' * 60)
print('ГИПОТЕЗА 2: регионы-лидеры > менее активные регионы')
print('=' * 60)

# Считаем число пользователей и долю возвратов по каждому региону
region_stats = user_profile.groupby('first_region').agg(
    users=('user_id', 'count'),
    return_rate=('is_two', 'mean')
).reset_index().sort_values('users', ascending=False).reset_index(drop=True)

# Определим «активные» регионы — топ-5 по числу пользователей
top5 = region_stats.head(5)
rest = region_stats.iloc[5:]

print('Топ-5 регионов по числу пользователей:')
print(top5.assign(return_rate_pct=lambda x: (x['return_rate']*100).round(2))
      [['first_region', 'users', 'return_rate_pct']].to_string(index=False))

rate_top5 = (top5['is_two'].sum() if 'is_two' in top5 else 
             (top5['return_rate'] * top5['users']).sum()) / top5['users'].sum()
rate_rest = (rest['return_rate'] * rest['users']).sum() / rest['users'].sum()

print(f'\nДоля возвратов в топ-5 регионах: {rate_top5*100:.2f}% (пользователей: {top5["users"].sum()})')
print(f'Доля возвратов в остальных регионах: {rate_rest*100:.2f}% (пользователей: {rest["users"].sum()})')
print(f'Разница: {(rate_top5 - rate_rest)*100:+.2f} п.п.')

if rate_top5 > rate_rest:
    print('→ В топ-5 регионах доля возвратов ВЫШЕ. Гипотеза подтверждается.')
else:
    print('→ В топ-5 регионах доля возвратов НЕ выше. Гипотеза НЕ подтверждается.')

# Корреляция между размером региона и долей возвратов
from scipy.stats import spearmanr
rho, p_corr = spearmanr(region_stats['users'], region_stats['return_rate'])
print(f'\nКорреляция Спирмена (размер региона ↔ доля возвратов): rho = {rho:.3f}, p = {p_corr:.4f}')
if p_corr < 0.05:
    print('Связь статистически значима.')
else:
    print('Связь статистически НЕ значима.')


In [ ]:
# ============================================================
# Визуализация отклонений
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гипотеза 1: отклонение от средней по типу мероприятия
event_stats = user_profile.groupby('first_event_type').agg(
    users=('user_id', 'count'),
    return_rate=('is_two', 'mean')
).reset_index()
event_stats['diff_pp'] = (event_stats['return_rate'] - base_rate) * 100
event_stats = event_stats.sort_values('diff_pp')

colors = ['crimson' if x < 0 else 'seagreen' for x in event_stats['diff_pp']]
axes[0].barh(event_stats['first_event_type'], event_stats['diff_pp'], color=colors)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Гипотеза 1: отклонение доли возвратов\nот средней по типу мероприятия, п.п.')
axes[0].set_xlabel('Отклонение от средней, п.п.')
for i, (v, n) in enumerate(zip(event_stats['diff_pp'], event_stats['users'])):
    axes[0].text(v + (0.2 if v >= 0 else -0.2), i, f'n={n}', va='center',
                 ha='left' if v >= 0 else 'right', fontsize=8)

# Гипотеза 2: размер региона vs доля возвратов
axes[1].scatter(region_stats['users'], region_stats['return_rate'] * 100, alpha=0.6)
axes[1].axhline(base_rate * 100, color='red', linestyle='--', label=f'Среднее: {base_rate*100:.1f}%')
axes[1].set_xscale('log')
axes[1].set_xlabel('Число пользователей в регионе (лог. шкала)')
axes[1].set_ylabel('Доля возвратов, %')
axes[1].set_title('Гипотеза 2: размер региона vs доля возвратов')
axes[1].legend()

plt.tight_layout()
plt.show()

Вывод по задаче 4.1.3.

    Гипотеза 1 (спорт vs концерты) — НЕ подтверждается. Пользователи, начавшие со спорта, возвращаются реже (55.8%), чем начавшие с концертов (61.8%). Разница −6.0 п.п. статистически значима (z = −3.35, p < 0.001). Спорт как «точка входа» уступает концертам по удержанию.

    Гипотеза 2 (регионы-лидеры) — подтверждается. В топ-5 регионах по числу пользователей доля возвратов 62.4% против 59.7% в остальных (+2.7 п.п.). Корреляция между размером региона и долей возвратов умеренная, но значимая (rho = 0.31, p = 0.005). При этом внутри топ-5 есть регионы с долей ниже средней (Озернинский край, Малиновоярский округ) — значит, размер региона не единственный фактор удержания.

Рекомендация для маркетинга: не стоит делать ставку на спорт как «канал входа» для удержания — лучше фокусироваться на театре, выставках и концертах. Региональные кампании в крупных регионах могут давать чуть более высокий возврат, но внутри них нужна дополнительная сегментация.

---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Разделяем пользователей на две группы
# ============================================================
one_order = user_profile[user_profile['is_two'] == 0]['avg_revenue_rub']
returned = user_profile[user_profile['is_two'] == 1]['avg_revenue_rub']

print(f'Пользователи с 1 заказом:      n = {len(one_order)}')
print(f'Вернувшиеся (2+ заказа):       n = {len(returned)}')

# ============================================================
# Описательная статистика по группам
# ============================================================
print('\n=== Статистика avg_revenue_rub ===')
stats_df = user_profile.groupby('is_two')['avg_revenue_rub'].describe()
stats_df.index = ['1 заказ', '2+ заказа']
print(stats_df.round(2))

# Дополнительно: медиана и перцентили
print('\n=== Перцентили ===')
for grp_name, grp in [('1 заказ', one_order), ('2+ заказа', returned)]:
    print(f'{grp_name}: '
          f'25% = {grp.quantile(0.25):.0f}, '
          f'50% = {grp.quantile(0.50):.0f}, '
          f'75% = {grp.quantile(0.75):.0f}, '
          f'90% = {grp.quantile(0.90):.0f}')

In [ ]:
# ============================================================
# Сравнительная гистограмма
# ============================================================
bins = np.linspace(0, 2628, 60)  # одинаковые интервалы

plt.figure(figsize=(12, 5))
plt.hist(one_order, bins=bins, alpha=0.5, density=True,
         label=f'1 заказ (n={len(one_order)})', color='steelblue')
plt.hist(returned, bins=bins, alpha=0.5, density=True,
         label=f'2+ заказа (n={len(returned)})', color='coral')
plt.xlabel('Средняя выручка с заказа, руб.')
plt.ylabel('Плотность')
plt.title('Распределение средней выручки с заказа:\n1 заказ vs 2+ заказа')
plt.legend()
plt.xlim(0, 2628)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Сравнение медиан и средних
# ============================================================
print('\n=== Сравнение медиан и средних ===')
print(f'1 заказ:    среднее = {one_order.mean():.2f}, медиана = {one_order.median():.2f}')
print(f'2+ заказа:  среднее = {returned.mean():.2f}, медиана = {returned.median():.2f}')
print(f'\nРазница средних:   {returned.mean() - one_order.mean():+.2f} руб.')
print(f'Разница медиан:    {returned.median() - one_order.median():+.2f} руб.')

Вывод по задаче 4.2.1. Средняя выручка с заказа у пользователей с 1 заказом и у вернувшихся (2+) практически не различается (545.8 vs 545.1 руб.). Однако медианная выручка у вернувшихся выше на 118 руб. (496.5 vs 378.9, +31%), а 25-й перцентиль — почти в 2 раза выше (271.7 vs 133.1). Это означает, что вернувшиеся пользователи совершают более дорогие типичные заказы и почти не покупают совсем дешёвые билеты, тогда как среди одноразовых много «разовых» дешёвых покупок, которые тянут среднее вниз, но не отражаются на медиане. Разброс у вернувшихся меньше (std 369 vs 519).

Интерпретация: сам по себе размер среднего чека не определяет возврат, но отсутствие очень дешёвых заказов и более высокий типичный чек связаны с повторными покупками. Для маркетинга это сигнал: пользователи, чей первый заказ дешёвый (до ~130 руб.), реже возвращаются — возможно, им стоит предлагать стимулы для второй покупки.

---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Разделяем на две группы
# ============================================================
grp_2_4 = user_profile[(user_profile['total_orders'] >= 2) & (user_profile['total_orders'] <= 4)]['avg_revenue_rub']
grp_5plus = user_profile[user_profile['total_orders'] >= 5]['avg_revenue_rub']

print(f'Группа 2–4 заказа: n = {len(grp_2_4)}')
print(f'Группа 5+ заказов:  n = {len(grp_5plus)}')

# ============================================================
# Описательная статистика
# ============================================================
print('\n=== Статистика avg_revenue_rub по группам ===')
summary = {
    '2–4 заказа': grp_2_4,
    '5+ заказов': grp_5plus
}
for name, grp in summary.items():
    print(f'\n{name}:')
    print(f'  n = {len(grp)}')
    print(f'  среднее = {grp.mean():.2f}')
    print(f'  медиана = {grp.median():.2f}')
    print(f'  std     = {grp.std():.2f}')
    print(f'  25%     = {grp.quantile(0.25):.2f}')
    print(f'  75%     = {grp.quantile(0.75):.2f}')
    print(f'  90%     = {grp.quantile(0.90):.2f}')

print(f'\nРазница медиан:  {grp_5plus.median() - grp_2_4.median():+.2f} руб.')
print(f'Разница средних: {grp_5plus.mean() - grp_2_4.mean():+.2f} руб.')

In [ ]:
# ============================================================
# Сравнительная гистограмма
# ============================================================
bins = np.linspace(0, 2628, 60)

plt.figure(figsize=(12, 5))
plt.hist(grp_2_4, bins=bins, alpha=0.5, density=True,
         label=f'2–4 заказа (n={len(grp_2_4)})', color='steelblue')
plt.hist(grp_5plus, bins=bins, alpha=0.5, density=True,
         label=f'5+ заказов (n={len(grp_5plus)})', color='seagreen')
plt.xlabel('Средняя выручка с заказа, руб.')
plt.ylabel('Плотность')
plt.title('Распределение средней выручки с заказа:\n2–4 заказа vs 5+ заказов')
plt.legend()
plt.xlim(0, 2628)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Боксплот для наглядности
# ============================================================
plt.figure(figsize=(8, 5))
plt.boxplot([grp_2_4, grp_5plus], labels=['2–4 заказа', '5+ заказов'])
plt.ylabel('Средняя выручка с заказа, руб.')
plt.title('Boxplot: средняя выручка с заказа')
plt.ylim(0, 2628)
plt.grid(alpha=0.3)
plt.show()

Вывод по задаче 4.2.2. Средняя выручка с заказа у групп 2–4 и 5+ заказов практически одинакова (552 vs 537 руб.). Однако медианная выручка у 5+ выше на 41 руб., 25-й перцентиль — в 1.5 раза выше (332 vs 220), а стандартное отклонение — в 1.4 раза ниже (299 vs 420). Это означает, что по мере роста числа заказов пользователи становятся более «ровными» покупателями: их типичный чек растёт, они почти не совершают очень дешёвых покупок, а разброс трат сокращается. Связь между лояльностью и средним чеком существует, но проявляется не в среднем, а в медиане и стабильности трат.

---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# 1. Распределение avg_tickets_count
# ============================================================
print('=== Статистика avg_tickets_count ===')
print(user_profile['avg_tickets_count'].describe())
print('\nПерцентили:')
for q in [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]:
    print(f'  {q*100:.0f}%: {user_profile["avg_tickets_count"].quantile(q):.2f}')

# Гистограмма
plt.figure(figsize=(10, 4))
user_profile['avg_tickets_count'].hist(bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Среднее количество билетов в заказе')
plt.ylabel('Число пользователей')
plt.title('Распределение пользователей по avg_tickets_count')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2. Сегментация по avg_tickets_count
# ============================================================
def tickets_segment(x):
    if x < 2:
        return '1–2'
    elif x < 3:
        return '2–3'
    elif x < 5:
        return '3–5'
    else:
        return '5+'

user_profile['tickets_segment'] = user_profile['avg_tickets_count'].apply(tickets_segment)

# Порядок сегментов для вывода
order = ['1–2', '2–3', '3–5', '5+']

seg_stats = user_profile.groupby('tickets_segment').agg(
    users=('user_id', 'count'),
    return_rate=('is_two', 'mean')
).reindex(order).reset_index()

seg_stats['share_pct'] = (seg_stats['users'] / seg_stats['users'].sum() * 100).round(2)
seg_stats['return_rate_pct'] = (seg_stats['return_rate'] * 100).round(2)

print('=== Сегменты по среднему количеству билетов ===')
print(seg_stats[['tickets_segment', 'users', 'share_pct', 'return_rate_pct']].to_string(index=False))

base_rate = user_profile['is_two'].mean() * 100
print(f'\nСредняя доля возвратов: {base_rate:.2f}%')

In [ ]:
# ============================================================
# 3. Визуализация: число пользователей и доля возвратов
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Число пользователей по сегментам
axes[0].bar(seg_stats['tickets_segment'], seg_stats['users'], color='steelblue')
axes[0].set_title('Число пользователей по сегментам')
axes[0].set_ylabel('Пользователей')
for i, v in enumerate(seg_stats['users']):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=9)

# Доля возвратов по сегментам
colors = ['crimson' if x < base_rate else 'seagreen' for x in seg_stats['return_rate_pct']]
axes[1].bar(seg_stats['tickets_segment'], seg_stats['return_rate_pct'], color=colors)
axes[1].axhline(base_rate, color='black', linestyle='--', label=f'Среднее: {base_rate:.1f}%')
axes[1].set_title('Доля пользователей с 2+ заказами')
axes[1].set_ylabel('Доля возвратов, %')
axes[1].legend()
for i, v in enumerate(seg_stats['return_rate_pct']):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

Вывод по задаче 4.2.3. Распределение пользователей по среднему количеству билетов сконцентрировано: 85.8% приходится на сегменты «2–3» (43.9%) и «3–5» (41.9%) билетов; «1–2» — 11.2%, «5+» — 3.1%.

Связь с возвратами выражена нелинейно:

    -«2–3 билета» — аномально высокая доля возвратов (73.6%), +12.2 п.п. к среднему. Это крупнейший сегмент — вероятно, типичный парный/семейный поход, формирующий привычку.

    -«5+ билетов» — аномально низкая доля (18.8%), −42.6 п.п. Сегмент малочисленный (3.1%), вероятно, представлен корпоративными/групповыми разовыми закупками.

    -«1–2» (51.2%) и «3–5» (54.3%) — ниже среднего, но без экстремумов.

Рекомендация: сегмент «2–3 билета» — приоритетный для удержания и персонализированных предложений. Сегмент «5+» требует отдельного анализа — возможно, это B2B-аудитория, для которой нужна другая стратегия.

---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# 1. Выделяем день недели первой покупки
# ============================================================
user_profile['first_dow'] = pd.to_datetime(user_profile['first_order_dt']).dt.dayofweek  # 0=Пн, 6=Вс

# Названия дней недели
dow_names = {0: 'Пн', 1: 'Вт', 2: 'Ср', 3: 'Чт', 4: 'Пт', 5: 'Сб', 6: 'Вс'}
user_profile['first_dow_name'] = user_profile['first_dow'].map(dow_names)

# ============================================================
# 2. Агрегация по дням недели
# ============================================================
dow_stats = user_profile.groupby(['first_dow', 'first_dow_name']).agg(
    users=('user_id', 'count'),
    return_rate=('is_two', 'mean')
).reset_index().sort_values('first_dow')

dow_stats['share_pct'] = (dow_stats['users'] / dow_stats['users'].sum() * 100).round(2)
dow_stats['return_rate_pct'] = (dow_stats['return_rate'] * 100).round(2)

base_rate = user_profile['is_two'].mean() * 100
dow_stats['diff_pp'] = (dow_stats['return_rate_pct'] - base_rate).round(2)

print('=== Первая покупка по дням недели ===')
print(f'Средняя доля возвратов: {base_rate:.2f}%\n')
print(dow_stats[['first_dow_name', 'users', 'share_pct', 'return_rate_pct', 'diff_pp']].to_string(index=False))

# ============================================================
# 3. Визуализация
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Число пользователей по дням недели
axes[0].bar(dow_stats['first_dow_name'], dow_stats['users'], color='steelblue')
axes[0].set_title('Число пользователей по дню недели первой покупки')
axes[0].set_ylabel('Пользователей')
for i, v in enumerate(dow_stats['users']):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=9)

# Доля возвратов по дням недели
colors = ['seagreen' if x > base_rate else 'crimson' for x in dow_stats['return_rate_pct']]
axes[1].bar(dow_stats['first_dow_name'], dow_stats['return_rate_pct'], color=colors)
axes[1].axhline(base_rate, color='black', linestyle='--', label=f'Среднее: {base_rate:.1f}%')
axes[1].set_title('Доля возвратов по дню недели первой покупки')
axes[1].set_ylabel('Доля 2+ заказов, %')
axes[1].legend()
for i, v in enumerate(dow_stats['return_rate_pct']):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# ============================================================
# 4. Проверка значимости различий (хи-квадрат)
# ============================================================
from scipy.stats import chi2_contingency

contingency = pd.crosstab(user_profile['first_dow_name'], user_profile['is_two'])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f'\nХи-квадрат: chi2 = {chi2:.2f}, p-value = {p:.4f}')
if p < 0.05:
    print('Различия по дням недели статистически значимы (p < 0.05).')
else:
    print('Различия по дням недели статистически НЕ значимы (p >= 0.05).')

Вывод по задаче 4.3.1. Пользователи приходят на платформу относительно равномерно по дням недели: от 12.8% (Вс) до 15.4% (Сб). Доля возвратов по дням варьируется слабо — от 59.5% (Чт) до 63.0% (Пн), разброс ~3.5 п.п. Различия статистически значимы (хи-квадрат, p = 0.007), но практический эффект мал: день недели первой покупки не является сильным фактором возврата. Небольшое преимущество у понедельника и субботы, небольшое отставание у четверга и пятницы — но эти различия не стоит переоценивать.

---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


In [ ]:
# ============================================================
# 1. Разделяем на группы и сравниваем avg_days_since_prev
# ============================================================
grp_2_4 = user_profile[(user_profile['total_orders'] >= 2) & 
                       (user_profile['total_orders'] <= 4)]['avg_days_since_prev'].dropna()
grp_5plus = user_profile[user_profile['total_orders'] >= 5]['avg_days_since_prev'].dropna()

print(f'Группа 2–4 заказа: n = {len(grp_2_4)}')
print(f'Группа 5+ заказов:  n = {len(grp_5plus)}')

print('\n=== Статистика avg_days_since_prev ===')
for name, grp in [('2–4 заказа', grp_2_4), ('5+ заказов', grp_5plus)]:
    print(f'\n{name}:')
    print(f'  среднее = {grp.mean():.2f}')
    print(f'  медиана = {grp.median():.2f}')
    print(f'  std     = {grp.std():.2f}')
    print(f'  25%     = {grp.quantile(0.25):.2f}')
    print(f'  75%     = {grp.quantile(0.75):.2f}')
    print(f'  90%     = {grp.quantile(0.90):.2f}')

print(f'\nРазница медиан:  {grp_5plus.median() - grp_2_4.median():+.2f} дн.')
print(f'Разница средних: {grp_5plus.mean() - grp_2_4.mean():+.2f} дн.')

# ============================================================
# 2. Визуализация: гистограмма и боксплот
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Гистограмма (ограничим 60 днями для наглядности)
bins = np.linspace(0, 60, 40)
axes[0].hist(grp_2_4, bins=bins, alpha=0.5, density=True,
             label=f'2–4 заказа (n={len(grp_2_4)})', color='steelblue')
axes[0].hist(grp_5plus, bins=bins, alpha=0.5, density=True,
             label=f'5+ заказов (n={len(grp_5plus)})', color='seagreen')
axes[0].set_xlabel('Средний интервал между заказами, дней')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Распределение среднего интервала')
axes[0].legend()
axes[0].set_xlim(0, 60)

# Боксплот
axes[1].boxplot([grp_2_4, grp_5plus], labels=['2–4 заказа', '5+ заказов'])
axes[1].set_ylabel('Средний интервал, дней')
axes[1].set_title('Boxplot: средний интервал между заказами')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

# ============================================================
# 3. Сегментация по интервалу и доля возвратов
# ============================================================
# Разобьём всех пользователей с 2+ заказами по интервалу
users_with_interval = user_profile[user_profile['avg_days_since_prev'].notna()].copy()

def interval_segment(x):
    if x < 7:
        return '< 7 дней'
    elif x < 14:
        return '7–14 дней'
    elif x < 30:
        return '14–30 дней'
    elif x < 60:
        return '30–60 дней'
    else:
        return '60+ дней'

users_with_interval['interval_segment'] = users_with_interval['avg_days_since_prev'].apply(interval_segment)

order = ['< 7 дней', '7–14 дней', '14–30 дней', '30–60 дней', '60+ дней']
int_stats = users_with_interval.groupby('interval_segment').agg(
    users=('user_id', 'count'),
    avg_orders=('total_orders', 'mean'),
    share_five=('is_five', 'mean')
).reindex(order).reset_index()

int_stats['share_pct'] = (int_stats['users'] / int_stats['users'].sum() * 100).round(2)
int_stats['share_five_pct'] = (int_stats['share_five'] * 100).round(2)

print('=== Сегменты по среднему интервалу между заказами ===')
print(int_stats[['interval_segment', 'users', 'share_pct', 'avg_orders', 'share_five_pct']].to_string(index=False))

# Визуализация
plt.figure(figsize=(10, 4))
plt.bar(int_stats['interval_segment'], int_stats['share_five_pct'], color='steelblue')
plt.axhline(user_profile['is_five'].mean() * 100, color='red', linestyle='--',
            label=f'Среднее 5+: {user_profile["is_five"].mean()*100:.1f}%')
plt.ylabel('Доля пользователей с 5+ заказами, %')
plt.title('Доля «лояльных» (5+ заказов) по среднему интервалу')
plt.legend()
plt.tight_layout()
plt.show()

Вывод по задаче 4.3.2. Средний интервал между заказами сильно различается у групп 2–4 и 5+: у лояльных (5+) он в 2 раза короче (9.9 vs 21.3 дня) и в 3.6 раза стабильнее (std 7.8 vs 28.5). У 2–4 заказов огромный разброс (90% = 59 дней), у 5+ — узкий (90% = 21 день).

Связь интервала с лояльностью нелинейна:

    7–14 дней — «золотой» ритм: доля 5+ максимальна (75.7%).

    <7 дней — 44.4% 5+: много «всплесков» без долгосрочной привычки.

    30+ дней — 9.3% и 0%: редкие покупки не приводят к лояльности.

Рекомендация: стимулировать регулярный возврат в течение 1–2 недель после покупки (напоминания, персональные предложения) — это наиболее перспективный путь к формированию лояльного ядра.

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
import phik
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Подготовка данных для корреляционного анализа
# ============================================================
# Отбираем признаки профиля пользователя для анализа
corr_cols = [
    'total_orders',
    'avg_revenue_rub',
    'avg_tickets_count',
    'avg_days_since_prev',
    'first_device',
    'first_region',
    'first_service',
    'first_event_type',
    'is_two',
    'is_five'
]

corr_df = user_profile[corr_cols].copy()

# Убедимся, что avg_days_since_prev не содержит пропусков (заполним медианой для корреляции)
corr_df['avg_days_since_prev'] = corr_df['avg_days_since_prev'].fillna(corr_df['avg_days_since_prev'].median())

# Определяем интервальные (числовые) колонки
interval_cols = ['avg_revenue_rub', 'avg_tickets_count', 'avg_days_since_prev', 'total_orders']

# ============================================================
# 2. Расчёт матрицы phik
# ============================================================
phik_matrix = corr_df.phik_matrix(interval_cols=interval_cols)

# Вывод корреляций с total_orders
print('=== Корреляция phik с total_orders ===')
print(phik_matrix['total_orders'].sort_values(ascending=False).round(3))

# ============================================================
# 3. Тепловая карта
# ============================================================
plt.figure(figsize=(10, 8))
sns.heatmap(phik_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=0, vmax=1, square=True)
plt.title('Матрица корреляции phik (полный профиль)')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4. Сегментация по total_orders и повторный анализ
# ============================================================
# Проверим разброс total_orders
print('=== Распределение total_orders ===')
print(user_profile['total_orders'].value_counts().head(10))
print(f'\nДоля пользователей с 1 заказом: {(user_profile["total_orders"] == 1).mean()*100:.1f}%')
print(f'Доля с 2–4 заказами: {((user_profile["total_orders"] >= 2) & (user_profile["total_orders"] <= 4)).mean()*100:.1f}%')
print(f'Доля с 5+ заказами: {(user_profile["total_orders"] >= 5).mean()*100:.1f}%')

# Создаём сегмент
def order_segment(x):
    if x == 1:
        return '1 заказ'
    elif x <= 4:
        return '2–4 заказа'
    else:
        return '5+ заказов'

user_profile['order_segment'] = user_profile['total_orders'].apply(order_segment)

# ============================================================
# 5. phik по сегментам (сравнение)
# ============================================================
# Убираем total_orders и бинарные признаки (они производные от него)
corr_cols_seg = [
    'avg_revenue_rub',
    'avg_tickets_count',
    'avg_days_since_prev',
    'first_device',
    'first_region',
    'first_service',
    'first_event_type'
]
interval_cols_seg = ['avg_revenue_rub', 'avg_tickets_count', 'avg_days_since_prev']

print('\n=== phik по сегментам (связь с order_segment) ===')
for seg in ['1 заказ', '2–4 заказа', '5+ заказов']:
    subset = user_profile[user_profile['order_segment'] == seg][corr_cols_seg].copy()
    subset['avg_days_since_prev'] = subset['avg_days_since_prev'].fillna(subset['avg_days_since_prev'].median())
    
    if len(subset) > 100:  # проверка размера
        phik_seg = subset.phik_matrix(interval_cols=interval_cols_seg)
        # Берём корреляции с avg_revenue_rub как пример (или можно с любым)
        print(f'\nСегмент «{seg}» (n={len(subset)}):')
        print(phik_seg.round(3))

Вывод по задаче 4.4.1. Корреляционный анализ phik показал, что с числом заказов наиболее связаны поведенческие признаки: среднее количество билетов (0.225), средняя выручка (0.219) и средний интервал между заказами (0.152). Признаки первого заказа (устройство, оператор, тип мероприятия, регион) имеют слабую связь (phik < 0.12) — точка входа почти не определяет число заказов.

Внутри сегментов по числу заказов (1, 2–4, 5+) сильная связь наблюдается между регионом и оператором (0.67–0.69) и умеренная — между типом мероприятия и регионом (0.48–0.54), что отражает географическую структуру бизнеса. Связь среднего чека и количества билетов умеренная (0.32–0.46).

Вывод для маркетинга: для прогнозирования числа заказов важнее поведение пользователя (как часто и на сколько он покупает), а не то, с какого устройства или мероприятия он начал. Это подтверждает результаты EDA: короткий регулярный интервал (7–14 дней) и стабильный средний чек — ключевые предикторы лояльности.

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

# Общие выводы и рекомендации

## Что мы сделали с данными

Мы взяли данные из базы сервиса Яндекс Афиша. В таблице было **292 034 заказа** и **22 000 пользователей**. Данные за период с июня по октябрь 2024 года.

Что мы сделали перед анализом:
- Оставили только покупки с телефонов (mobile) и компьютеров (desktop). Остальные устройства (планшеты и т.п.) были очень редкими.
- Убрали фильмы, потому что по ним было мало данных.
- Перевели выручку из тенге в рубли. Часть заказов была оплачена в тенге, и мы посчитали их в рублях по курсу на день покупки.
- Убрали выбросы по выручке (самые дорогие заказы, выше 99-го перцентиля) и отрицательную выручку (такое бывает при возвратах).
- Убрали «странных» пользователей с огромным числом заказов (больше 152). Похоже, это были боты или корпоративные аккаунты.

После всей чистки осталось **287 405 заказов** и **21 838 пользователей**. Потом мы убрали ещё 216 пользователей-аномалий, и для анализа осталось **21 622 пользователя**.

## Что мы узнали о пользователях

- **61,3% пользователей** делают 2 и больше заказа (возвращаются).
- **28,3% пользователей** делают 5 и больше заказов (это самые лояльные).
- Средняя выручка с одного заказа — примерно **545 рублей**.
- Обычно покупают **2–3 билета** за раз.

## Какие признаки первого заказа связаны с возвратом

- **Тип мероприятия:** чаще возвращаются те, кто начал с **театра** или **выставок**. А вот те, кто начал со **спорта**, возвращаются реже (55,8% против 61,8% у концертов). Гипотеза про спорт **не подтвердилась**.
- **Устройство:** те, кто купил первый раз с **компьютера**, возвращаются чуть чаще, чем с телефона (63,8% против 60,8%).
- **Регион:** в крупных регионах (где больше всего пользователей) доля возвратов немного выше, чем в маленьких (62,4% против 59,7%). Но внутри крупных регионов тоже есть свои «хорошие» и «плохие» (например, Озернинский край — всего 55,3%).
- **Оператор:** разница между операторами небольшая (60–65%), сильно никто не выделяется.

## Как выручка и количество билетов связаны с возвратом

- **Средняя выручка** сама по себе почти не отличается у тех, кто вернулся, и у тех, кто нет (545 против 545). Но у вернувшихся **медиана выше** (496 против 379). Это значит, что у вернувшихся почти нет очень дешёвых заказов, а у «одноразовых» — много.
- **Количество билетов:** самый интересный результат. Лучше всего возвращаются те, кто покупает в среднем **2–3 билета** (возврат 73,6%!). А те, кто покупает **5+ билетов**, почти не возвращаются (всего 18,8%). Но таких пользователей мало, так что результат может быть не очень точным.

## Что влияет на удержание из временных характеристик

- **День недели** первой покупки почти не важен. Разница между днями маленькая (59–63%), хотя формально она значима. Сильно опираться на это не стоит.
- **Интервал между покупками** — вот это важно! Те, кто делает **5+ заказов**, возвращаются в среднем каждые **10 дней**, а те, кто 2–4 заказа — каждые **21 день**. Лучше всего возвращаются те, кто покупает **раз в 7–14 дней** (возврат 75,7%).

## Что связано с числом покупок (корреляция)

С числом заказов больше всего связаны:
- **Среднее количество билетов** (связь 0,225).
- **Средняя выручка** (0,219).
- **Средний интервал между заказами** (0,152).

А вот признаки первого заказа (устройство, оператор, тип мероприятия, регион) связаны с числом заказов **очень слабо** (меньше 0,12). То есть неважно, с чего человек начал — важно, как он покупает дальше.

## Рекомендации для маркетинга

1. **Обратить внимание на тех, кто покупает 2–3 билета.** Это самый большой сегмент (около 44% пользователей), и у них самая высокая доля возвратов (73,6%). Их стоит удерживать и предлагать им персональные условия.
2. **Стимулировать возврат в течение 1–2 недель после первой покупки.** Именно этот ритм (7–14 дней) сильнее всего связан с лояльностью. Можно отправлять напоминания или давать скидку на второй заказ.
3. **Не делать ставку на спорт как на «точку входа».** Люди, которые начинают со спорта, возвращаются реже, чем те, кто начинает с концертов, театра или выставок. Лучше привлекать через театр и выставки.
4. **Осторожно с теми, кто покупает 5+ билетов сразу.** Они почти не возвращаются (18,8%). Скорее всего, это корпоративные или групповые закупки. Для них нужна отдельная стратегия (например, B2B-предложения).
5. **Пользователи с очень дешёвыми первыми заказами (до 130 рублей) возвращаются реже.** Возможно, им стоит предлагать бонус или скидку на вторую покупку, чтобы «зацепить».

## Ограничения анализа

- Данные только за 5 месяцев (июнь–октябрь 2024), поэтому нельзя сказать, что будет дальше. Есть сезонность (осенью заказов больше).
- Мы убрали «странных» пользователей с огромным числом заказов, но, возможно, часть из них — реальные корпоративные клиенты.
- В некоторых сегментах мало пользователей (например, «ёлки» или операторы с малым числом заказов), поэтому их проценты могут быть неточными.

### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.

**Вставьте ссылку на проект в этой ячейке тетрадки перед отправкой проекта на ревью.**

Репозиторий проекта: https://github.com/dmortar/2-tickets